In [6]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import threading
import time
import random
import math
import csv
from concurrent.futures import ThreadPoolExecutor

# ==========================================
# MODULE 1: ML & MATHEMATICAL BACKEND
# ==========================================

def calculate_rmse(R_actual, P, Q):
    """
    Calculates the Root Mean Square Error (RMSE) between actual ratings 
    and the model's predicted ratings. Tracks individual errors for distribution plotting.
    """
    users, items, K = len(R_actual), len(R_actual[0]), len(P[0])
    error_sum, count = 0.0, 0
    errors = [] 
    
    for i in range(users):
        for j in range(items):
            if R_actual[i][j] > 0:
                pred = sum(P[i][k] * Q[k][j] for k in range(K))
                err = R_actual[i][j] - pred
                error_sum += err ** 2
                errors.append(err)
                count += 1
                
    rmse = math.sqrt(error_sum / count) if count > 0 else 0
    return rmse, errors

def _compute_row(i, A, B, C, cols_A, cols_B):
    """Helper function to compute a single row dot-product workload."""
    for j in range(cols_B):
        C[i][j] = sum(A[i][k] * B[k][j] for k in range(cols_A))

def multiply_sequential(A, B):
    """Standard O(N^3) Matrix Multiplication executed on a single thread."""
    C = [[0.0] * len(B[0]) for _ in range(len(A))]
    for i in range(len(A)):
        _compute_row(i, A, B, C, len(A[0]), len(B[0]))
    return C

def multiply_one_thread_per_row(A, B):
    """Coarse-grained parallelism: Spawns one OS thread per row in Matrix A."""
    C = [[0.0] * len(B[0]) for _ in range(len(A))]
    threads = [threading.Thread(target=_compute_row, args=(i, A, B, C, len(A[0]), len(B[0]))) for i in range(len(A))]
    for t in threads: t.start()
    for t in threads: t.join()
    return C

def multiply_thread_pool(A, B, max_workers):
    """Production standard parallelism: Distributes row workloads across a fixed pool of hardware threads."""
    C = [[0.0] * len(B[0]) for _ in range(len(A))]
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for i in range(len(A)):
            executor.submit(_compute_row, i, A, B, C, len(A[0]), len(B[0]))
    return C


# ==========================================
# MODULE 2: GRAPHICAL USER INTERFACE
# ==========================================

class RecommenderDashboardV4:
    def __init__(self, root):
        self.root = root
        self.root.title("Enterprise ML Recommender & Analytics Suite")
        self.root.geometry("1200x900")
        self.root.configure(bg="#121212")
        
        # --- UI Styling Configuration ---
        style = ttk.Style()
        style.theme_use('clam')
        style.configure("TFrame", background="#1e1e1e")
        style.configure("TLabel", background="#1e1e1e", foreground="#ffffff", font=("Segoe UI", 10))
        style.configure("TLabelframe", background="#1e1e1e", foreground="#00ffcc", font=("Segoe UI", 10, "bold"))
        style.configure("TLabelframe.Label", background="#1e1e1e", foreground="#00ffcc")
        style.configure("TButton", background="#007acc", foreground="#ffffff", font=("Segoe UI", 10, "bold"))
        style.map("TButton", background=[("active", "#005f9e")], foreground=[("disabled", "#aaaaaa")])
        
        # --- Data State Variables ---
        self.R, self.R_train, self.R_test = [], [], []
        self.P, self.Q = [], [] 
        self.active_model = "SGD"
        
        self.rating_counts = {1:0, 2:0, 3:0, 4:0, 5:0}
        self.total_cells = 0
        self.filled_cells = 0
        self.loss_history = []
        
        self._build_ui()

    def _build_ui(self):
        # Global Notebook
        notebook = ttk.Notebook(self.root)
        notebook.pack(fill="both", expand=True, padx=10, pady=10)
        
        self.tab_data = ttk.Frame(notebook)
        self.tab_train = ttk.Frame(notebook)
        self.tab_benchmark = ttk.Frame(notebook)
        
        notebook.add(self.tab_data, text=" 📊 1. Data Insights ")
        notebook.add(self.tab_train, text=" 🧠 2. ML Training & Eval ")
        notebook.add(self.tab_benchmark, text=" ⏱️ 3. Engine Benchmark ")

        self._build_data_tab()
        self._build_train_tab()
        self._build_benchmark_tab()

        # Global Status Bar
        self.status_var = tk.StringVar()
        self.status_var.set("System Ready. Please load a dataset to begin.")
        status_bar = ttk.Label(self.root, textvariable=self.status_var, background="#0a0a0a", foreground="#aaaaaa", anchor="w", padding=(10, 5))
        status_bar.pack(side="bottom", fill="x")

    def _update_status(self, message):
        """Helper to update the global status bar."""
        self.status_var.set(message)
        self.root.update_idletasks()

    # ==========================================
    # TAB 1: DATA INGESTION & INSIGHTS
    # ==========================================
    def _build_data_tab(self):
        ctrl_frame = ttk.LabelFrame(self.tab_data, text="Step 1: Dataset Ingestion (CSV)", padding=15)
        ctrl_frame.pack(fill="x", padx=20, pady=10)
        
        ttk.Button(ctrl_frame, text="Browse & Load Dataset", command=self.load_csv).pack(side="left", padx=10)
        self.lbl_data_info = ttk.Label(ctrl_frame, text="Awaiting CSV file (Format: userId, movieId, rating)...", foreground="#ffcc00")
        self.lbl_data_info.pack(side="left", padx=20)
        
        charts_container = ttk.Frame(self.tab_data)
        charts_container.pack(fill="both", expand=True, padx=20, pady=10)
        
        frame_hist = ttk.LabelFrame(charts_container, text="Data Skew: Rating Distribution", padding=10)
        frame_hist.pack(side="left", fill="both", expand=True, padx=(0, 10))
        self.canvas_hist = tk.Canvas(frame_hist, bg="#2d2d2d", highlightthickness=0)
        self.canvas_hist.pack(fill="both", expand=True)
        
        frame_pie = ttk.LabelFrame(charts_container, text="Cold Start Problem: Matrix Sparsity", padding=10)
        frame_pie.pack(side="right", fill="both", expand=True, padx=(10, 0))
        self.canvas_pie = tk.Canvas(frame_pie, bg="#2d2d2d", highlightthickness=0)
        self.canvas_pie.pack(fill="both", expand=True)

    def load_csv(self):
        filepath = filedialog.askopenfilename(filetypes=[("CSV Files", "*.csv")])
        if not filepath: return
        
        self._update_status("Parsing CSV file...")
        user_map, item_map = {}, {}
        raw_data = []
        self.rating_counts = {1:0, 2:0, 3:0, 4:0, 5:0}
        
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                reader = csv.reader(f)
                next(reader, None) # Skip Header
                for row in reader:
                    if len(row) < 3: continue
                    u, i, r = row[0], row[1], round(float(row[2]))
                    r = max(1, min(r, 5)) # Clamp ratings between 1 and 5
                    
                    # Map arbitrary string IDs to contiguous array indices
                    if u not in user_map: user_map[u] = len(user_map)
                    if i not in item_map: item_map[i] = len(item_map)
                    
                    raw_data.append((user_map[u], item_map[i], r))
                    self.rating_counts[r] += 1
                    
            # Memory cap for GUI responsiveness (600x1000 matrix max)
            # Heavy Stress Test Mode
            U, M = min(len(user_map), 2000), min(len(item_map), 3000)
            self.R = [[0.0] * M for _ in range(U)]
            self.filled_cells = 0
            
            for u, i, r in raw_data:
                if u < U and i < M:
                    self.R[u][i] = r
                    self.filled_cells += 1
                    
            self.total_cells = U * M
            self.lbl_data_info.config(text=f"Loaded Dataset: {U} Users × {M} Movies | Total Known Ratings: {self.filled_cells:,}")
            self._draw_histogram()
            self._draw_pie_chart()
            self._update_status("CSV successfully loaded. Proceed to ML Training tab.")
            
        except Exception as e:
            messagebox.showerror("File Error", f"Could not process the CSV file.\nError: {e}")
            self._update_status("Error loading dataset.")

    def _draw_histogram(self):
        self.canvas_hist.delete("all")
        w, h = 400, 250
        max_val = max(self.rating_counts.values()) or 1
        colors = ["#ff4d4d", "#ff9933", "#ffcc00", "#99cc33", "#33cc33"]
        
        for i in range(1, 6):
            val = self.rating_counts[i]
            bar_h = (val / max_val) * (h - 60)
            x0 = i * (w / 6) - 25
            y0 = h - 30 - bar_h
            x1 = x0 + 50
            y1 = h - 30
            self.canvas_hist.create_rectangle(x0, y0, x1, y1, fill=colors[i-1])
            self.canvas_hist.create_text((x0+x1)/2, y1 + 15, text=f"{i} ★", fill="white", font=("Segoe UI", 10, "bold"))
            # Annotate bar values
            self.canvas_hist.create_text((x0+x1)/2, y0 - 10, text=str(val), fill="#aaaaaa", font=("Segoe UI", 8))

    def _draw_pie_chart(self):
        self.canvas_pie.delete("all")
        if self.total_cells == 0: return
        fill_deg = (self.filled_cells / self.total_cells) * 360
        self.canvas_pie.create_arc(100, 30, 300, 230, start=0, extent=fill_deg, fill="#00ffcc", outline="")
        self.canvas_pie.create_arc(100, 30, 300, 230, start=fill_deg, extent=360-fill_deg, fill="#444444", outline="")
        
        density = (self.filled_cells / self.total_cells) * 100
        self.canvas_pie.create_text(200, 260, text=f"Data Density: {density:.2f}%", fill="#00ffcc", font=("Segoe UI", 12, "bold"))
        self.canvas_pie.create_text(200, 280, text="Recommendation algorithms must interpolate the missing data.", fill="#aaaaaa", font=("Segoe UI", 9, "italic"))

    # ==========================================
    # TAB 2: TRAINING & EVALUATION
    # ==========================================
    def _build_train_tab(self):
        ctrl_frame = ttk.LabelFrame(self.tab_train, text="Step 2: Model Architecture & Execution", padding=15)
        ctrl_frame.pack(fill="x", padx=20, pady=10)
        
        ttk.Label(ctrl_frame, text="Select Algorithm Topology:").grid(row=0, column=0, padx=5)
        self.model_var = tk.StringVar(value="Latent Factor Model (SGD)")
        
        # highly descriptive dropdown options
        algorithms = [
            "Latent Factor Model (SGD) - Best for Sparse Data", 
            "User-User Collaborative (Cosine) - Maps Similar Users", 
            "Item-Item Collaborative (Cosine) - Maps Similar Movies"
        ]
        combo = ttk.Combobox(ctrl_frame, textvariable=self.model_var, values=algorithms, state="readonly", width=55)
        combo.grid(row=0, column=1, padx=10)
        
        self.btn_train = ttk.Button(ctrl_frame, text="Initialize Pipeline", command=self.start_training)
        self.btn_train.grid(row=0, column=2, padx=20)
        
        self.lbl_train_status = ttk.Label(self.tab_train, text="Select an algorithm and begin training...", font=("Consolas", 12))
        self.lbl_train_status.pack(pady=10)
        
        eval_container = ttk.Frame(self.tab_train)
        eval_container.pack(fill="both", expand=True, padx=20, pady=10)
        
        frame_loss = ttk.LabelFrame(eval_container, text="Real-Time Gradient Descent Loss Curve", padding=10)
        frame_loss.pack(side="left", fill="both", expand=True, padx=(0, 10))
        self.canvas_loss = tk.Canvas(frame_loss, bg="#2d2d2d", highlightthickness=0)
        self.canvas_loss.pack(fill="both", expand=True)

        frame_err = ttk.LabelFrame(eval_container, text="Model Evaluation: Outlier Detection (Test Set)", padding=10)
        frame_err.pack(side="right", fill="both", expand=True, padx=(10, 0))
        self.canvas_err = tk.Canvas(frame_err, bg="#2d2d2d", highlightthickness=0)
        self.canvas_err.pack(fill="both", expand=True)

    def start_training(self):
        if not self.R: 
            messagebox.showerror("Empty Dataset", "Please load a CSV dataset in Tab 1 first.")
            return
            
        self._update_status("Splitting dataset 80/20 for Train/Test validation...")
        users, items = len(self.R), len(self.R[0])
        
        # 80/20 Train Test Split
        self.R_train = [[self.R[i][j] if random.random() < 0.8 else 0 for j in range(items)] for i in range(users)]
        self.R_test = [[self.R[i][j] if self.R_train[i][j] == 0 else 0 for j in range(items)] for i in range(users)]
        
        self.btn_train.config(state="disabled")
        self.canvas_loss.delete("all")
        self.canvas_err.delete("all")
        
        sel = self.model_var.get()
        if "Latent Factor" in sel:
            self.active_model = "SGD"
            threading.Thread(target=self._train_sgd, daemon=True).start()
        elif "User-User" in sel:
            self.active_model = "USER_CF"
            threading.Thread(target=self._train_cf, args=("USER",), daemon=True).start()
        elif "Item-Item" in sel:
            self.active_model = "ITEM_CF"
            threading.Thread(target=self._train_cf, args=("ITEM",), daemon=True).start()

    def _train_sgd(self):
        self._update_status("Training Latent Factor Model using Stochastic Gradient Descent...")
        users, items, K, lr = len(self.R_train), len(self.R_train[0]), 10, 0.005
        self.P = [[random.uniform(0.1, 0.5) for _ in range(K)] for _ in range(users)]
        self.Q = [[random.uniform(0.1, 0.5) for _ in range(items)] for _ in range(K)]
        self.loss_history = []
        
        for step in range(20):
            for i in range(users):
                for j in range(items):
                    if self.R_train[i][j] > 0:
                        dot = sum(self.P[i][k] * self.Q[k][j] for k in range(K))
                        e = self.R_train[i][j] - dot
                        for k in range(K):
                            self.P[i][k] += lr * (2 * e * self.Q[k][j] - 0.02 * self.P[i][k])
                            self.Q[k][j] += lr * (2 * e * self.P[i][k] - 0.02 * self.Q[k][j])
            
            rmse, _ = calculate_rmse(self.R_train, self.P, self.Q)
            self.loss_history.append(rmse)
            self.root.after(0, self._draw_loss_curve)
            self.root.after(0, lambda e=step+1, r=rmse: self.lbl_train_status.config(text=f"Optimization Epoch {e}/20 | Training RMSE: {r:.4f}"))
            
        test_rmse, errors = calculate_rmse(self.R_test, self.P, self.Q)
        self.root.after(0, lambda: self.lbl_train_status.config(text=f"✓ Training Converged! Validation RMSE (Unseen Data): {test_rmse:.4f}", foreground="#00ffcc"))
        self.root.after(0, lambda: self._draw_error_dist(errors))
        self.root.after(0, lambda: self.btn_train.config(state="normal"))
        self._update_status("Model trained successfully. Proceed to Engine Benchmark.")

    def _train_cf(self, mode):
        self.root.after(0, lambda: self.lbl_train_status.config(text="Computing massive Similarity Matrix (O(N^3) operation)..."))
        self._update_status(f"Building {mode} Collaborative Filtering Matrix...")
        
        R_T = [[self.R_train[j][i] for j in range(len(self.R_train))] for i in range(len(self.R_train[0]))]
        
        t0 = time.perf_counter()
        if mode == "USER":
            multiply_thread_pool(self.R_train, R_T, 12) 
        else:
            multiply_thread_pool(R_T, self.R_train, 12) 
            
        dur = time.perf_counter() - t0
        self.root.after(0, lambda: self.lbl_train_status.config(text=f"✓ Collaborative Matrix Computed in {dur:.2f} seconds.", foreground="#00ffcc"))
        self.root.after(0, lambda: self.btn_train.config(state="normal"))
        self._update_status("Matrix computed. Proceed to Engine Benchmark.")

    def _draw_loss_curve(self):
        self.canvas_loss.delete("all")
        if len(self.loss_history) < 2: return
        w, h = 400, 200
        max_r, min_r = max(self.loss_history), min(self.loss_history)
        rng = max_r - min_r if max_r != min_r else 1
        coords = []
        for i, val in enumerate(self.loss_history):
            x = (i / 19) * (w - 40) + 20
            y = h - 20 - ((val - min_r) / rng) * (h - 40)
            coords.extend([x, y])
        self.canvas_loss.create_line(*coords, fill="#00ffcc", width=3, smooth=True)

    def _draw_error_dist(self, errors):
        self.canvas_err.delete("all")
        if not errors: return
        w, h = 400, 200
        bins = {"Perfect":0, "Minor Error":0, "Moderate":0, "Severe":0}
        for e in errors:
            e = abs(e)
            if e < 0.2: bins["Perfect"] += 1
            elif e < 1.0: bins["Minor Error"] += 1
            elif e < 2.0: bins["Moderate"] += 1
            else: bins["Severe"] += 1
            
        max_b = max(bins.values()) or 1
        labels = list(bins.keys())
        colors = ["#33cc33", "#99cc33", "#ff9933", "#ff4d4d"]
        
        for i, label in enumerate(labels):
            val = bins[label]
            bar_h = (val / max_b) * (h - 50)
            x0 = i * (w / 4) + 10
            y0 = h - 30 - bar_h
            x1 = x0 + (w / 4) - 20
            y1 = h - 30
            self.canvas_err.create_rectangle(x0, y0, x1, y1, fill=colors[i])
            self.canvas_err.create_text((x0+x1)/2, y1 + 15, text=label, fill="white", font=("Segoe UI", 9))
            self.canvas_err.create_text((x0+x1)/2, y0 - 10, text=f"{val}", fill="#aaaaaa", font=("Segoe UI", 8))

    # ==========================================
    # TAB 3: BENCHMARK
    # ==========================================
    def _build_benchmark_tab(self):
        frame_bench = ttk.LabelFrame(self.tab_benchmark, text="Step 3: Multithreading CPU Stress Test", padding=15)
        frame_bench.pack(fill="x", padx=20, pady=10)
        
        ttk.Label(frame_bench, text="Concurrent Worker Threads:").grid(row=0, column=0)
        
        # Set default to 12 threads for optimal execution on modern multi-core architectures
        self.scl_threads = ttk.Scale(frame_bench, from_=2, to=24, value=12, command=lambda v: self.lbl_tval.config(text=f"{int(float(v))} threads"))
        self.scl_threads.grid(row=0, column=1, padx=10)
        self.lbl_tval = ttk.Label(frame_bench, text="12 threads", foreground="#ffcc00", font=("Segoe UI", 10, "bold"))
        self.lbl_tval.grid(row=0, column=2, padx=(0, 20))
        
        self.btn_bench = ttk.Button(frame_bench, text="Execute CPU Benchmark", command=self.run_benchmark)
        self.btn_bench.grid(row=0, column=3, padx=20)
        
        bench_container = ttk.Frame(self.tab_benchmark)
        bench_container.pack(fill="both", expand=True, padx=20, pady=10)
        
        self.txt_bench = tk.Text(bench_container, height=8, bg="#0a0a0a", fg="#ffffff", font=("Consolas", 11), borderwidth=1, relief="solid")
        self.txt_bench.pack(fill="x", pady=(0, 10))
        
        frame_chart = ttk.LabelFrame(bench_container, text="Parallel Execution Topology Analysis (Lower is Better)", padding=10)
        frame_chart.pack(fill="both", expand=True)
        self.canvas_bench = tk.Canvas(frame_chart, bg="#2d2d2d", highlightthickness=0)
        self.canvas_bench.pack(fill="both", expand=True)

    def run_benchmark(self):
        if not self.R_train: 
            return messagebox.showwarning("Prerequisite Missing", "Please execute the ML Pipeline in Tab 2 before benchmarking.")
            
        self.btn_bench.config(state="disabled")
        self.txt_bench.delete("1.0", "end")
        self.canvas_bench.delete("all")
        self._update_status("Running CPU Benchmarks. Please wait...")
        threading.Thread(target=self._bench_worker, daemon=True).start()

    def _bench_worker(self):
        workers = int(float(self.scl_threads.get()))
        
        if self.active_model == "SGD":
            self.root.after(0, lambda: self.txt_bench.insert("end", f"▶ Workload Mode: Latent Factor Inference\n  Operation: Matrix P (Users) × Matrix Q^T (Items)\n\n"))
            A = self.P
            B = [[self.Q[j][i] for j in range(len(self.Q))] for i in range(len(self.Q[0]))]
        elif self.active_model == "USER_CF":
            self.root.after(0, lambda: self.txt_bench.insert("end", f"▶ Workload Mode: User Similarity Mapping\n  Operation: Matrix R (Users) × Matrix R^T (Users)\n\n"))
            A = self.R_train
            B = [[self.R_train[j][i] for j in range(len(self.R_train))] for i in range(len(self.R_train[0]))]
        else: # ITEM_CF
            self.root.after(0, lambda: self.txt_bench.insert("end", f"▶ Workload Mode: Item Similarity Mapping\n  Operation: Matrix R^T (Items) × Matrix R (Items)\n\n"))
            A = [[self.R_train[j][i] for j in range(len(self.R_train))] for i in range(len(self.R_train[0]))]
            B = self.R_train

        results = {}
        
        # 1. Sequential Execution
        t0 = time.perf_counter()
        multiply_sequential(A, B)
        results["Sequential"] = time.perf_counter() - t0
        self.root.after(0, lambda: self.txt_bench.insert("end", f"  ↳ [Single-Threaded] Sequential Loop:     {results['Sequential']:.4f} sec\n"))
            
        # 2. Native Threading (One Per Row)
        t0 = time.perf_counter()
        multiply_one_thread_per_row(A, B)
        results["Thread-per-Row"] = time.perf_counter() - t0
        self.root.after(0, lambda: self.txt_bench.insert("end", f"  ↳ [Coarse-Grained]  Thread-per-Row:      {results['Thread-per-Row']:.4f} sec\n"))
            
        # 3. Thread Pool Architecture
        t0 = time.perf_counter()
        multiply_thread_pool(A, B, max_workers=workers)
        results["Thread Pool"] = time.perf_counter() - t0
        self.root.after(0, lambda: self.txt_bench.insert("end", f"  ↳ [Managed Block]   Thread Pool ({workers:02d}):  {results['Thread Pool']:.4f} sec\n"))
        
        self.root.after(0, lambda: self._draw_bench_chart(results))
        self.root.after(0, lambda: self.btn_bench.config(state="normal"))
        self._update_status("Benchmark calculations complete.")

    def _draw_bench_chart(self, results):
        self.canvas_bench.delete("all")
        w, h = 800, 200
        max_t = max(results.values()) or 1
        colors = ["#ff4d4d", "#ffcc00", "#00ffcc"]
        labels = list(results.keys())
        
        for i, label in enumerate(labels):
            val = results[label]
            bar_w = (val / max_t) * (w - 250)
            y0 = i * 50 + 20
            
            # Label
            self.canvas_bench.create_text(200, y0 + 15, text=label, fill="white", anchor="e", font=("Segoe UI", 11, "bold"))
            # Bar
            self.canvas_bench.create_rectangle(210, y0, 210 + bar_w, y0 + 30, fill=colors[i], outline="")
            # Value Annotation
            self.canvas_bench.create_text(220 + bar_w, y0 + 15, text=f"{val:.3f} s", fill="white", anchor="w", font=("Segoe UI", 10))

if __name__ == "__main__":
    root = tk.Tk()
    app = RecommenderDashboardV4(root)
    root.mainloop()